# GPT-Style LLM Pipeline

Main goal: don't focus code — we want to **understand and interpret the outputs**.

This notebook targets common sticking points:
- **Tokens vs token IDs** (and why we need both)
- **Embeddings** + **positional information**
- **Q, K, V** (what they mean and why we project into them)
- **Softmax inside attention** (why it produces weights; why scaling matters)
- **Causal masking** (why GPT can’t “peek” at the future)
- **Sliding window** (two meanings: *data sampling* vs *local attention*)
- **Attention heads** (why multi-head helps)
- **Logits** (what the model outputs) and **why generation is one-by-one**

**How to use it**
- Before running each section, read the **WHY** bullets.
- After running, look at the printed tables/heatmaps and ask: *“What changed, and why?”*


## The main pipeline steps

A GPT-style model is basically this loop:

1. **Encode** text → token IDs
2. **Embed** IDs → vectors (and add positional information)
3. Transformer blocks mix information using **masked self-attention**
4. A final linear layer produces **logits** (one score per vocab token)
5. Convert logits → probabilities (softmax) and pick the next token
6. Append the token and repeat

Remember:
- In **one forward pass (training)**, computations across all tokens are mostly **parallel** (matrix multiplications).
- During **generation (inference)**, the *outer* loop is **sequential** (you add **one** new token per step).


In [ ]:
# Make sure Python can import the helper files shipped with this notebook
import sys, os
from pathlib import Path
# Ensure local helper files (llm_pipeline_utils/) are importable
sys.path.insert(0, str(Path.cwd()))

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from llm_pipeline_utils.viz_utils import set_seed, fmt_matrix, plot_heatmap
from llm_pipeline_utils.tokenizer_toy import (
    regex_tokenize,
    train_toy_bpe,
    build_vocab,
    encode,
    decode,
)
from llm_pipeline_utils.attention_toy import (
    scaled_dot_product_attention,
    make_causal_mask,
    make_sliding_window_causal_mask,
    MultiHeadSelfAttention,
)
from llm_pipeline_utils.toy_gpt import TinyGPT, cross_entropy_next_token

set_seed(0)
print('torch:', torch.__version__)


## 1) Raw text → tokens

**WHY this step exists**
- Neural networks operate on **numbers**, not raw strings.
- Tokenization decides the *units* the model will learn over (words, subwords, punctuation).

**What to look for**
- The same sentence can become very different token sequences depending on the tokenizer.


In [ ]:
sample_text = "Every effort moves you. Every effort matters!"

tokens_regex = regex_tokenize(sample_text)
print('Raw text:')
print(sample_text)
print('\nRegex tokens:')
print(tokens_regex)
print('\n#tokens:', len(tokens_regex))


### 1.1) Subword tokenization (toy BPE)

**WHY BPE/subwords are popular**
- “Unknown words” are common in real text. Subwords let you build new words from familiar pieces.
- They keep the vocabulary size manageable while still handling many word forms.

This is a **toy** BPE implementation so you can *see* the merges. (Real GPT tokenizers are more complex.)


In [ ]:
corpus = [
    "Every effort moves you.",
    "Effort matters.",
    "Effortful people move mountains.",
]

bpe = train_toy_bpe(corpus, num_merges=30)
print('First 10 learned merges:')
print(bpe.merges[:10])

tokens_bpe = bpe.tokenize(sample_text)
print('\nToy-BPE tokens:')
print(tokens_bpe)
print('\n#tokens:', len(tokens_bpe))


## 2) Tokens → token IDs (a vocabulary)

**WHY this step exists**
- A model can’t store weights for string tokens; it stores weights for **integer IDs**.
- A **vocabulary** is the shared dictionary: `token ↔ id`.

**What to look for**
- Encoding is deterministic given a vocabulary.
- Decoding is the reverse mapping.


In [ ]:
# We'll build a vocab from our toy-BPE tokens.
vocab = build_vocab(tokens_bpe)

ids = encode(tokens_bpe, vocab)
print('Vocab size:', vocab.size)
print('\nTokens -> IDs:')
for t, i in list(zip(tokens_bpe, ids))[:20]:
    print(f"{t:>10} -> {i}")

print('\nDecode back (approx formatting):')
print(decode(ids, vocab))


## 3) Token IDs → embeddings (vectors)

**WHY this step exists**
- IDs are just labels. The model needs **vectors** to do math.
- An embedding layer is a trainable lookup table of shape `[vocab_size, d_model]`.

**Key idea**
- Same token ID → same embedding vector (before attention).


In [ ]:
d_model = 8
emb = nn.Embedding(vocab.size, d_model)

x_ids = torch.tensor(ids[:8])  # take first 8 tokens
x_emb = emb(x_ids)             # [T, d_model]

print('x_ids shape:', x_ids.shape)
print('x_emb shape:', x_emb.shape)

# Show two identical tokens get identical embeddings
# (if they appear in our first 8 tokens)
for tok in set(tokens_bpe[:8]):
    idxs = [i for i,t in enumerate(tokens_bpe[:8]) if t==tok]
    if len(idxs) >= 2:
        a, b = idxs[:2]
        print(f"\nToken '{tok}' appears at positions {a} and {b}")
        print('Embedding difference norm:', torch.norm(x_emb[a]-x_emb[b]).item())
        break


### 3.1) Positional embeddings (order information)

**WHY this step exists**
- Attention is *content-based*. Without extra info, it doesn’t know token order.
- GPT-style models often add a learned **positional embedding** for each position.

**What to look for**
- The *same token* at two positions gets different final input vectors after adding position.


In [ ]:
T = x_ids.shape[0]
pos_emb = nn.Embedding(64, d_model)  # pretend max context is 64

pos = torch.arange(T)
final_inp = x_emb + pos_emb(pos)  # [T, d_model]

print('final_inp shape:', final_inp.shape)

# Demonstrate: if a token repeats, adding position changes its vector
for tok in set(tokens_bpe[:8]):
    idxs = [i for i,t in enumerate(tokens_bpe[:8]) if t==tok]
    if len(idxs) >= 2:
        a, b = idxs[:2]
        print(f"Token '{tok}' at positions {a} and {b}")
        print('Token-embedding diff norm:', torch.norm(x_emb[a]-x_emb[b]).item())
        print('Final-input diff norm:', torch.norm(final_inp[a]-final_inp[b]).item())
        break


## 4) Sliding window **data sampling** (next-token prediction)

**WHY this step exists**
- GPT-style pretraining is usually **next-token prediction**.
- For input sequence `x`, the target `y` is the same sequence shifted left by one.
- A sliding window creates *many* training examples from one long stream.

**Important:** This is a *data* sliding window (not the same as sliding-window attention).


In [ ]:
def make_next_token_pairs(token_ids, seq_len=8, stride=4):
    xs, ys = [], []
    for start in range(0, len(token_ids) - seq_len - 1, stride):
        x = token_ids[start : start + seq_len]
        y = token_ids[start + 1 : start + seq_len + 1]
        xs.append(x)
        ys.append(y)
    return xs, ys

# Make a longer stream by repeating our sample
stream = encode(bpe.tokenize(' '.join(corpus) * 3), vocab)
xs, ys = make_next_token_pairs(stream, seq_len=8, stride=4)

print('How many training samples from one stream?', len(xs))

# Show a few (decoded)
for i in range(3):
    x, y = xs[i], ys[i]
    print('\nSample', i)
    print('x:', decode(x, vocab))
    print('y:', decode(y, vocab))


## 5) Self-attention: the core computation

Self-attention answers: **“For this token, which other tokens should I use as context?”**

### 5.1 Simplified attention

**WHY this step exists**
- Token meaning depends on context (e.g., “bank” in “river bank” vs “bank robbery”).
- Attention lets each position *mix* information from other positions.

We’ll start with the *idea*:
- Compute **similarity scores** between tokens
- Normalize them into **weights** (softmax)
- Take a **weighted sum** to form a new representation


In [ ]:
# We'll use tiny made-up embeddings so we can inspect everything.
# Think of each row as one token vector.

tokens = ['Every', 'effort', 'moves', 'you', '.']
X = torch.tensor([
    [ 1.0,  0.0,  0.0,  0.0],  # Every
    [ 0.9,  0.1,  0.0,  0.0],  # effort (similar to Every)
    [ 0.0,  1.0,  0.0,  0.0],  # moves
    [ 0.0,  0.9,  0.1,  0.0],  # you (similar to moves)
    [ 0.0,  0.0,  0.0,  1.0],  # .
])

# Simplified: Q=K=V=X
out = scaled_dot_product_attention(Q=X, K=X, V=X, mask=None)

print('Scores (QK^T)')
print(fmt_matrix(out.scores, row_labels=tokens, col_labels=tokens))

print('\nAttention weights = softmax(scores / sqrt(d_k))')
print(fmt_matrix(out.attn_weights, row_labels=tokens, col_labels=tokens, fmt='{:.3f}'))

plot_heatmap(out.attn_weights, xlabels=tokens, ylabels=tokens, title='Attention weights (rows sum to 1)')


### 5.2 Softmax (and scaling) inside attention

**WHY softmax?**
- Turns arbitrary scores into **nonnegative weights** that sum to 1 per row.
- Makes the operation a stable “mixture”: output = weighted average of values.

**WHY divide by √d?**
- Dot products grow with dimension.
- Without scaling, scores get large → softmax becomes extremely peaked → gradients can get unstable.

We’ll *measure* that effect quickly.


In [ ]:
# Softmax + scaling demo
# Goal: see that without scaling, dot-product scores get large as dimension grows,
# making softmax very "peaky" (one position dominates).

def peakiness(weights: torch.Tensor) -> float:
    # Average of the maximum attention weight per row (higher = peakier)
    return float(weights.max(dim=-1).values.mean())

for d in [4, 16, 64, 256]:
    set_seed(0)
    T = 10
    Q = torch.randn(T, d)
    K = torch.randn(T, d)

    scores = Q @ K.T
    w_noscale = torch.softmax(scores, dim=-1)
    w_scaled  = torch.softmax(scores / math.sqrt(d), dim=-1)

    print(f"d={d:3d} | peakiness(no scale)={peakiness(w_noscale):.3f} | peakiness(scaled)={peakiness(w_scaled):.3f}")

print("Interpretation: as d grows, scaling keeps softmax from becoming too extreme.")


## 6) Q, K, V: making attention *trainable*

In real transformers we do not set Q=K=V=X.

**WHY project into Q, K, V?**
- It gives the model **learnable knobs** for:
  - what a token is “asking for” (**Query**)
  - what a token “offers” (**Key**)
  - what information actually gets passed along (**Value**)

Formulas (per token):
- `Q = X Wq` , `K = X Wk` , `V = X Wv`

We’ll compute them and inspect shapes.


In [ ]:
set_seed(1)
T, d_in, d_k, d_v = 5, 4, 4, 4

X = torch.randn(T, d_in)
Wq = torch.randn(d_in, d_k)
Wk = torch.randn(d_in, d_k)
Wv = torch.randn(d_in, d_v)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

print('Shapes')
print('X:', tuple(X.shape), 'Q:', tuple(Q.shape), 'K:', tuple(K.shape), 'V:', tuple(V.shape))

attn = scaled_dot_product_attention(Q=Q, K=K, V=V)
print('\nAttention weights (random, but inspectable):')
print(attn.attn_weights)


## 7) Causal (masked) attention

**WHY masking?**
- GPT is **autoregressive**: it predicts the next token from previous tokens.
- During training, we must prevent the model from “cheating” by looking at future tokens.

Causal mask rule:
- Position *i* may attend to positions `0..i` only.

We’ll apply the mask and verify that future weights become ~0.


In [ ]:
tokens = ['Every', 'effort', 'moves', 'you', '.']
X = torch.randn(len(tokens), 4)

mask = make_causal_mask(T=len(tokens), device=X.device)

attn_full = scaled_dot_product_attention(Q=X, K=X, V=X, mask=None)
attn_masked = scaled_dot_product_attention(Q=X, K=X, V=X, mask=mask)

print('Causal mask (True = allowed):')
print(mask.int())

print('\nMasked attention weights (upper triangle should be ~0):')
print(fmt_matrix(attn_masked.attn_weights, row_labels=tokens, col_labels=tokens, fmt='{:.3f}'))

plot_heatmap(attn_masked.attn_weights, xlabels=tokens, ylabels=tokens, title='Causal attention weights')


## 8) Sliding window **attention** (local + causal)

Be careful: **"sliding window" can mean two different things**:

1. **Data sampling sliding window** (Section 4): how we create many (x, y) examples.
2. **Attention sliding window** (this section): a model design choice to reduce cost.

**WHY local attention?**
- Full attention is O(T²) per layer.
- For very long contexts, some models restrict attention to a recent window.

Local causal rule (window = W):
- token *i* can attend to `max(0, i-W+1) .. i`.


In [ ]:
T = 12
window = 4

X = torch.randn(T, 8)
mask_local = make_sliding_window_causal_mask(T, window=window, device=X.device)

attn_local = scaled_dot_product_attention(Q=X, K=X, V=X, mask=mask_local)

print('Local causal mask (1=allowed):')
print(mask_local.int())

plot_heatmap(attn_local.attn_weights, xlabels=[str(i) for i in range(T)], ylabels=[str(i) for i in range(T)],
             title=f'Sliding-window causal attention (window={window})')


## 9) Multi-head attention (what changes with heads?)

**WHY multiple heads?**
- A single attention matrix is one “view” of relevance.
- Multiple heads let the model learn **different relevance patterns at once**.

Mechanically:
- Split `d_model` into `n_heads` smaller subspaces.
- Each head has its own Q, K, V projections.
- Compute attention per head, then concatenate.

We’ll create a *controlled* example where head 1 and head 2 focus on different features.


In [ ]:
# Controlled toy example
# d_model=4, n_heads=2 -> each head sees 2 dims.

set_seed(0)

tokens = ['Alice', 'likes', 'Bob', 'because', 'Bob', 'smiles']
T = len(tokens)

# Build token vectors with TWO kinds of features:
# - dims 0-1: "who" features (entity identity)
# - dims 2-3: "relation" features (syntax-ish)
X = torch.tensor([
    [1.0, 0.0, 0.0, 0.0],  # Alice: who-feature A
    [0.0, 0.0, 1.0, 0.0],  # likes: relation feature 1
    [0.8, 0.2, 0.0, 0.0],  # Bob: who-feature B (similar to itself)
    [0.0, 0.0, 0.7, 0.3],  # because: relation mix
    [0.8, 0.2, 0.0, 0.0],  # Bob again
    [0.0, 0.0, 0.2, 0.8],  # smiles: relation feature 2
])

mha = MultiHeadSelfAttention(d_model=4, n_heads=4, dropout=0.0)

# Force Wq/Wk/Wv to act like identity (so heads attend based on raw features)
with torch.no_grad():
    mha.Wq.weight.copy_(torch.eye(4))
    mha.Wk.weight.copy_(torch.eye(4))
    mha.Wv.weight.copy_(torch.eye(4))
    mha.Wo.weight.copy_(torch.eye(4))

mask = make_causal_mask(T)

out, attn = mha(X[None, :, :], mask=mask)  # add batch dim
attn = attn[0]  # [nH,T,T]

for h in range(attn.shape[0]):
    plot_heatmap(attn[h], xlabels=tokens, ylabels=tokens, title=f'Head {h} attention (causal)')


## 10) Logits (the thing the model actually outputs)

At each position *t*, the model produces **one score per vocabulary token**.
Those raw scores are called **logits**.

**WHY logits (instead of probabilities directly)?**
- Probabilities must be in [0,1] and sum to 1 — that constraint makes training numerically tricky.
- Most losses (cross-entropy) are implemented in a **numerically stable** way using logits.

Key facts:
- Logits can be any real numbers (negative/positive).
- `softmax(logits)` turns them into probabilities.
- Adding the same constant to every logit **does not change** the softmax probabilities.


In [ ]:
logits = torch.tensor([2.0, 1.0, 0.0, -1.0])
probs = torch.softmax(logits, dim=-1)

print('logits:', logits.tolist())
print('softmax probs:', probs.tolist())
print('sum probs:', probs.sum().item())

# Invariance to adding a constant
shifted = logits + 100.0
print('\nsoftmax(logits) == softmax(logits+100)?', torch.allclose(torch.softmax(logits, -1), torch.softmax(shifted, -1)))

# Temperature: lower temperature -> peakier distribution
for temp in [2.0, 1.0, 0.5]:
    p = torch.softmax(logits / temp, dim=-1)
    print(f"temp={temp}: probs={p.tolist()}")


### 10.1) Cross-entropy uses logits

When we say “predict the next token”, training compares:
- model’s **logits** at each position vs.
- the **true next token ID**

PyTorch’s `cross_entropy` expects logits and does the stable math internally.


In [ ]:
# One training example: 4-class classification
true_class = torch.tensor([0])  # the correct next-token id
loss = F.cross_entropy(logits.view(1, -1), true_class)
print('Cross-entropy loss:', float(loss))

# If the correct class logit increases, loss decreases
logits_better = logits.clone()
logits_better[0] += 2.0
loss_better = F.cross_entropy(logits_better.view(1, -1), true_class)
print('Loss after boosting correct logit:', float(loss_better))


## 11) Parallel vs one-by-one tokens

Two different questions get mixed up:

### A) Inside a forward pass (training *and* inference)
- The model takes a sequence of length **T** and produces logits for **all T positions** in one go.
- That’s mostly parallel computation.

### B) During text generation (inference loop)
- You don’t know the future tokens yet.
- So you generate **one new token**, append it, and run another forward pass.

We’ll demonstrate both with a tiny GPT-style model.


In [ ]:
# Build a tiny vocab from our toy-BPE tokens
vocab = build_vocab(bpe.tokenize(' '.join(corpus) * 5))

# Tiny GPT (random weights for now)
model = TinyGPT(vocab_size=vocab.size, block_size=16, d_model=32, n_layers=2, n_heads=4, dropout=0.0)

prompt = "Every effort"
prompt_ids = encode(bpe.tokenize(prompt), vocab)
idx = torch.tensor(prompt_ids, dtype=torch.long)[None, :]  # [B=1,T]

out = model(idx)
print('Input shape:', tuple(idx.shape))
print('Logits shape:', tuple(out.logits.shape), ' = [B, T, vocab_size]')

# "Parallel" next-token predictions for every position in the input
pred_ids_each_pos = out.logits.argmax(dim=-1)[0].tolist()
print('\nPredicted token at each position (argmax per position):')
for t in range(idx.shape[1]):
    print(f"pos {t}: input='{vocab.id_to_token[idx[0,t].item()]}' -> predicts '{vocab.id_to_token[pred_ids_each_pos[t]]}'")

# For generation, we only use the *last* position's logits
next_id = out.logits[0, -1].argmax().item()
print('\nNext token for generation (from last position only):', vocab.id_to_token[next_id])


## Summary

- **Tokens**: discrete text units the model reads
- **Token IDs**: integers for tokens (via a vocabulary)
- **Embeddings**: vectors for token IDs (trainable lookup table)
- **Positional embeddings**: add order information
- **Q, K, V**: learned projections that make attention trainable
- **Attention scores**: `QKᵀ` (similarity)
- **Scaled scores**: divide by `√d_k` for stability
- **Softmax**: converts scores into weights that sum to 1
- **Mask**: blocks illegal attention links (e.g., future tokens)
- **Heads**: multiple parallel attention patterns, concatenated
- **Logits**: raw scores over vocab; `softmax(logits)` → probabilities
- **Training**: adjusts weights so correct next-token logits get higher
- **Generation**: outer loop is one token at a time, but each forward pass is parallel over the context